# Modelo Causal v6 — IPTW com novas variáveis

Extensão do notebook 05 com as variáveis derivadas no EDA (`causal_model_00_eda.ipynb`).

Baseado no artigo: *Causal Inference with DoWhy #4 — IPTW*  
https://medium.com/data-science-explained/causal-inference-with-dowhy-4-inverse-probability-of-treatment-weighting-iptw-817de28e541f

## Novidades em relação ao v5

| Item | v5 | v6 |
|---|---|---|
| Tratamentos | `despacho_lento`, `aprovacao_lenta` | + `frete_alto`, `pedido_grande` |
| Outcomes | `entrega_atrasada`, `OSR` | + `review_positivo` |
| Combinações | 3 | 8 |

## Combinações analisadas

| # | Tratamento | Outcome | Assoc. bruta (EDA) |
|---|---|---|---|
| 1 | `despacho_lento` | `entrega_atrasada` | +0.079 |
| 2 | `despacho_lento` | `review_positivo` | −0.086 |
| 3 | `aprovacao_lenta` | `entrega_atrasada` | +0.016 |
| 4 | `aprovacao_lenta` | `review_positivo` | −0.016 |
| 5 | `frete_alto` | `entrega_atrasada` | +0.003 |
| 6 | `frete_alto` | `review_positivo` | −0.005 |
| 7 | `pedido_grande` | `entrega_atrasada` | −0.016 |
| 8 | `pedido_grande` | `review_positivo` | −0.163 |

## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

from app.config.settings import INTERIM_DATA_DIR

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Carregamento e preparação dos dados

In [ ]:
df = pd.read_parquet(os.path.join(INTERIM_DATA_DIR, "interim_dataset.parquet"))

# Converter datas
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Intervalos operacionais
df["approval_time_hours"] = (
    df["order_approved_at"] - df["order_purchase_timestamp"]
).dt.total_seconds() / 3600

df["dispatch_time_days"] = (
    df["order_delivered_carrier_date"] - df["order_approved_at"]
).dt.days

df["delay_days"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

df["perc_freight"] = df["total_freight"] / df["total_price"]

# --- Tratamentos (variáveis binárias) ---
df["despacho_lento"]  = (df["dispatch_time_days"] > 3).astype(int)
df["aprovacao_lenta"] = (df["approval_time_hours"] > 24).astype(int)
df["frete_alto"]      = (df["perc_freight"] > 0.2).astype(int)
df["pedido_grande"]   = (df["n_items"] > 1).astype(int)

# --- Outcomes ---
df["entrega_atrasada"] = (df["delay_days"] > 0).astype(int)
df["review_positivo"]  = (df["review_score"] >= 4).astype(int)

print(f"Shape: {df.shape}")
print("\nPrevalência dos tratamentos:")
for t in ["despacho_lento", "aprovacao_lenta", "frete_alto", "pedido_grande"]:
    print(f"  {t}: {df[t].mean()*100:.1f}%")
print("\nPrevalência dos outcomes:")
for o in ["entrega_atrasada", "review_positivo", "OSR"]:
    print(f"  {o}: {df[o].mean()*100:.1f}%")

## 3. Confundidores

Variáveis que afetam **tanto o tratamento quanto o outcome** e precisam ser controladas.

| Confundidor | Justificativa |
|---|---|
| `total_price` | Produto caro → frete proporcionalmente menor; vendedor maior → despacho mais rápido |
| `n_items` | Pedido grande → mais complexidade logística |
| `avg_weight` | Produto pesado → frete mais alto; pode atrasar |
| `customer_state` | Estado distante → frete alto e entrega mais lenta |
| `purchase_month` | Sazonalidade → datas de pico aumentam atrasos |
| `purchase_weekday` | Dia da semana afeta aprovação e despacho |
| `purchase_hour` | Hora da compra afeta tempo de aprovação |
| `n_items_missing_info` | Dados faltantes no pedido podem indicar sellers problemáticos |

In [ ]:
CONFUNDIDORES = [
    "total_price",
    "n_items",
    "avg_weight",
    "customer_state",
    "purchase_month",
    "purchase_weekday",
    "purchase_hour",
    "n_items_missing_info"
]

## 4. Funções IPTW

In [ ]:
def preprocess(df, treatment, outcome, confundidores):
    cols = confundidores + [treatment, outcome]
    df_out = df[cols].dropna().copy()

    le = LabelEncoder()
    df_out["customer_state"] = le.fit_transform(df_out["customer_state"])

    continuous_cols = [c for c in confundidores if c != "customer_state"]
    scaler = StandardScaler()
    df_out[continuous_cols] = scaler.fit_transform(df_out[continuous_cols])

    return df_out


def compute_iptw_weights(df_model, treatment, confundidores, stabilized=True):
    """
    Calcula pesos IPTW estabilizados.
    - Tratados (T=1): w = P(T=1) / e(X)
    - Controles (T=0): w = P(T=0) / (1 - e(X))
    """
    X = df_model[confundidores].values
    T = df_model[treatment].values

    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X, T)
    ps = lr.predict_proba(X)[:, 1]
    ps = np.clip(ps, 0.01, 0.99)

    p_treated = T.mean()
    weights = np.where(
        T == 1,
        (p_treated / ps) if stabilized else (1 / ps),
        ((1 - p_treated) / (1 - ps)) if stabilized else (1 / (1 - ps))
    )
    return ps, weights


def compute_smd(df_model, treatment, confundidores, weights=None):
    """SMD < 0.1 indica bom balanço entre grupos."""
    smds = {}
    T = df_model[treatment].values
    for col in confundidores:
        x = df_model[col].values
        if weights is not None:
            mean1 = np.average(x[T == 1], weights=weights[T == 1])
            mean0 = np.average(x[T == 0], weights=weights[T == 0])
            var1  = np.average((x[T == 1] - mean1)**2, weights=weights[T == 1])
            var0  = np.average((x[T == 0] - mean0)**2, weights=weights[T == 0])
        else:
            mean1, var1 = x[T == 1].mean(), x[T == 1].var()
            mean0, var0 = x[T == 0].mean(), x[T == 0].var()
        pooled_std = np.sqrt((var1 + var0) / 2)
        smds[col] = abs(mean1 - mean0) / pooled_std if pooled_std > 0 else 0
    return pd.Series(smds)


def ate_iptw(df_model, treatment, outcome, weights):
    """ATE = média ponderada tratados - média ponderada controles."""
    T = df_model[treatment].values
    Y = df_model[outcome].values
    return (
        np.average(Y[T == 1], weights=weights[T == 1]) -
        np.average(Y[T == 0], weights=weights[T == 0])
    )


def bootstrap_ci(df_model, treatment, outcome, confundidores,
                 n_bootstrap=500, alpha=0.05):
    ates = []
    n = len(df_model)
    for _ in range(n_bootstrap):
        sample = df_model.sample(n=n, replace=True)
        _, w = compute_iptw_weights(sample, treatment, confundidores)
        ates.append(ate_iptw(sample, treatment, outcome, w))
    lower = np.percentile(ates, 100 * alpha / 2)
    upper = np.percentile(ates, 100 * (1 - alpha / 2))
    return lower, upper, np.array(ates)

In [ ]:
def run_iptw_analysis(df, treatment, outcome, confundidores,
                      assoc_bruta=None, n_bootstrap=500):
    print(f"{'='*60}")
    print(f"IPTW: {treatment} → {outcome}")
    print(f"{'='*60}")

    df_model = preprocess(df, treatment, outcome, confundidores)
    print(f"N após dropna: {len(df_model):,}")

    # Propensity scores e pesos
    ps, weights = compute_iptw_weights(df_model, treatment, confundidores)
    print(f"\nPropensity score — média: {ps.mean():.3f} | "
          f"min: {ps.min():.3f} | max: {ps.max():.3f}")
    print(f"Pesos IPTW        — média: {weights.mean():.3f} | "
          f"min: {weights.min():.3f} | max: {weights.max():.3f}")

    # Balanço das covariáveis
    smd_antes  = compute_smd(df_model, treatment, confundidores)
    smd_depois = compute_smd(df_model, treatment, confundidores, weights=weights)
    df_smd = pd.DataFrame({
        "SMD antes":  smd_antes.round(4),
        "SMD depois": smd_depois.round(4),
        "Balanceado?": smd_depois.apply(lambda x: "✅" if x < 0.1 else "❌")
    })
    print("\n--- Balanço das covariáveis (SMD < 0.1 = balanceado) ---")
    print(df_smd.to_string())

    # ATE
    ate = ate_iptw(df_model, treatment, outcome, weights)
    print(f"\nATE (IPTW): {ate:.6f}")
    if assoc_bruta is not None:
        print(f"Assoc. bruta (EDA): {assoc_bruta:+.4f}")

    # Bootstrap IC
    print(f"Calculando IC via bootstrap ({n_bootstrap} amostras)...")
    lower, upper, boot_ates = bootstrap_ci(
        df_model, treatment, outcome, confundidores, n_bootstrap
    )
    print(f"IC 95%: [{lower:.6f}, {upper:.6f}]")
    significativo = ("✅ Significativo"
                     if (lower > 0 or upper < 0)
                     else "❌ Não significativo (IC contém zero)")
    print(f"Resultado: {significativo}")

    # Gráfico bootstrap
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Plot 1: distribuição bootstrap do ATE
    axes[0].hist(boot_ates, bins=40, color="steelblue", alpha=0.7, edgecolor="white")
    axes[0].axvline(ate,   color="red",    linewidth=2,   label=f"ATE = {ate:.4f}")
    axes[0].axvline(lower, color="orange", linewidth=1.5, linestyle="--",
                    label=f"IC inf = {lower:.4f}")
    axes[0].axvline(upper, color="orange", linewidth=1.5, linestyle="--",
                    label=f"IC sup = {upper:.4f}")
    axes[0].axvline(0,     color="black",  linewidth=1,   linestyle=":")
    axes[0].set_title(f"Bootstrap ATE\n{treatment} → {outcome}")
    axes[0].set_xlabel("ATE estimado")
    axes[0].set_ylabel("Frequência")
    axes[0].legend(fontsize=8)
    axes[0].grid(axis="y", linestyle="--", alpha=0.4)

    # Plot 2: SMD antes vs. depois
    x = np.arange(len(confundidores))
    axes[1].barh(x - 0.2, smd_antes.values,  0.4, label="Antes IPTW",  color="coral",     alpha=0.8)
    axes[1].barh(x + 0.2, smd_depois.values, 0.4, label="Depois IPTW", color="steelblue", alpha=0.8)
    axes[1].axvline(0.1, color="red", linestyle="--", linewidth=1, label="Limite 0.1")
    axes[1].set_yticks(x)
    axes[1].set_yticklabels(confundidores, fontsize=8)
    axes[1].set_xlabel("SMD")
    axes[1].set_title("Balanço das covariáveis")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    fname = f"../../reports/figures/iptw_v6_{treatment}_{outcome}.png"
    plt.savefig(fname, dpi=150)
    plt.show()

    return {
        "treatment": treatment, "outcome": outcome,
        "N": len(df_model),
        "assoc_bruta": assoc_bruta,
        "ate": ate, "ic_lower": lower, "ic_upper": upper,
        "significativo": (lower > 0 or upper < 0)
    }

## 5. Análises IPTW

### 5.1 — Despacho lento → Entrega atrasada

In [ ]:
r1 = run_iptw_analysis(df, "despacho_lento", "entrega_atrasada",
                       CONFUNDIDORES, assoc_bruta=+0.0793)

### 5.2 — Despacho lento → Review positivo

In [ ]:
r2 = run_iptw_analysis(df, "despacho_lento", "review_positivo",
                       CONFUNDIDORES, assoc_bruta=-0.0858)

### 5.3 — Aprovação lenta → Entrega atrasada

In [ ]:
r3 = run_iptw_analysis(df, "aprovacao_lenta", "entrega_atrasada",
                       CONFUNDIDORES, assoc_bruta=+0.0160)

### 5.4 — Aprovação lenta → Review positivo

In [ ]:
r4 = run_iptw_analysis(df, "aprovacao_lenta", "review_positivo",
                       CONFUNDIDORES, assoc_bruta=-0.0159)

### 5.5 — Frete alto → Entrega atrasada

In [ ]:
r5 = run_iptw_analysis(df, "frete_alto", "entrega_atrasada",
                       CONFUNDIDORES, assoc_bruta=+0.0033)

### 5.6 — Frete alto → Review positivo

In [ ]:
r6 = run_iptw_analysis(df, "frete_alto", "review_positivo",
                       CONFUNDIDORES, assoc_bruta=-0.0051)

### 5.7 — Pedido grande → Entrega atrasada

In [ ]:
r7 = run_iptw_analysis(df, "pedido_grande", "entrega_atrasada",
                       CONFUNDIDORES, assoc_bruta=-0.0157)

### 5.8 — Pedido grande → Review positivo

In [ ]:
r8 = run_iptw_analysis(df, "pedido_grande", "review_positivo",
                       CONFUNDIDORES, assoc_bruta=-0.1633)

## 6. Comparação final: Assoc. bruta (EDA) vs. ATE IPTW

## 7. Interpretação dos resultados

### 7.1 Qualidade do ajuste por tratamento

| Tratamento | Pesos estáveis? | Balanço atingido? | Confiabilidade |
|---|---|---|---|
| `despacho_lento` | ✅ max=2.78 | ✅ todos SMD < 0.1 | **Alta** |
| `aprovacao_lenta` | ✅ max=2.24 | ✅ todos SMD < 0.1 | **Alta** |
| `frete_alto` | ❌ max=55.4 | ❌ `total_price`, `avg_weight`, `customer_state` | **Baixa** |
| `pedido_grande` | ✅ max=0.92 | ❌ `n_items` (SMD=1.93), `total_price` | **Baixa** |

---

### 7.2 Achados por análise

#### ✅ Confiáveis

**`despacho_lento → entrega_atrasada` | ATE = +0.079 | IC [0.073, 0.084]**
- Efeito causal mais forte e robusto do modelo
- Despacho lento **aumenta em ~8 p.p.** a probabilidade de entrega atrasada
- Viés dos confundidores era mínimo (−0.0002): a associação bruta já refletia o efeito real
- Balanço perfeito: `avg_weight` era o confundidor mais relevante (SMD 0.22 → 0.005)

**`despacho_lento → review_positivo` | ATE = −0.088 | IC [−0.095, −0.081]**
- Despacho lento **reduz em ~9 p.p.** a chance de review positivo
- Segundo maior efeito confiável do modelo
- Confirma que o cliente percebe e penaliza atrasos no despacho na avaliação

**`aprovacao_lenta → entrega_atrasada` | ATE = +0.016 | IC [0.012, 0.021]**
- Efeito real positivo: aprovação lenta propaga atraso no ciclo do pedido
- Menor que o despacho — aprovação é o início do ciclo, impacta indiretamente

**`aprovacao_lenta → review_positivo` | ATE = −0.010 | IC [−0.017, −0.004]**
- A assoc. bruta (−0.016) estava inflada: confundidores respondiam por 35% do efeito observado
- Efeito real é fraco — cliente não percebe diretamente o tempo de aprovação

---

#### ⚠️ Baixa confiabilidade

**`frete_alto → entrega_atrasada` | ATE = +0.014 | IC [0.002, 0.024]**
- Balanço não atingido: `total_price` (SMD=0.30) e `avg_weight` (SMD piorou: 0.16→0.21)
- Peso máximo = 55.4 → instabilidade severa
- Estimativa **não confiável** — confundimento residual alto

**`frete_alto → review_positivo` | ATE = −0.050 | IC [−0.069, −0.030]**
- Assoc. bruta era −0.005, IPTW sugere −0.050: diferença de 10× indica que o modelo
  não conseguiu remover o viés adequadamente (confundidores não balanceados)
- Estimativa **não confiável**

**`pedido_grande → entrega_atrasada` | ATE = −0.016 | IC [−0.021, −0.012]**
- Problema estrutural: `n_items` (base do tratamento) está na lista de confundidores
- SMD de `n_items` = 1.93 antes **e depois** do IPTW — o ajuste não surtiu efeito
- Estimativa com confundimento residual

**`pedido_grande → review_positivo` | ATE = −0.169 | IC [−0.180, −0.159]**
- Maior efeito bruto da tabela, porém com o mesmo problema estrutural
- O efeito pode refletir que pedidos com mais itens têm maior probabilidade de
  ao menos 1 item gerar insatisfação — mas o modelo não isola isso adequadamente

---

### 7.3 Próximos passos recomendados

| Problema | Solução proposta |
|---|---|
| `frete_alto`: regressão logística não balanceia | Usar propensity score com Random Forest ou GBM |
| `frete_alto`: `total_price` colinear com tratamento | Remover `total_price` dos confundidores e usar `avg_price` |
| `pedido_grande`: `n_items` = base do tratamento | Remover `n_items` da lista de confundidores e rodar novamente |

In [ ]:
resultados = [r1, r2, r3, r4, r5, r6, r7, r8]

df_comp = pd.DataFrame(resultados)
df_comp["vies"] = df_comp["ate"] - df_comp["assoc_bruta"]
df_comp["sig"] = df_comp["significativo"].map({True: "✅", False: "❌"})

print("Assoc. bruta (EDA) vs ATE IPTW ajustado por confundidores")
print("=" * 80)
cols_show = ["treatment", "outcome", "N", "assoc_bruta", "ate",
             "ic_lower", "ic_upper", "vies", "sig"]
print(df_comp[cols_show].rename(columns={
    "treatment": "Tratamento",
    "outcome":   "Outcome",
    "assoc_bruta": "Assoc. bruta",
    "ate":       "ATE IPTW",
    "ic_lower":  "IC inf",
    "ic_upper":  "IC sup",
    "vies":      "Viés removido",
    "sig":       "Significativo?"
}).to_string(index=False))

In [ ]:
# Gráfico comparativo: assoc. bruta vs ATE IPTW
labels = [f"{r['treatment']}\n→ {r['outcome']}" for r in resultados]
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, df_comp["assoc_bruta"], width,
               label="Assoc. bruta (EDA)", color="coral", alpha=0.8)
bars2 = ax.bar(x + width/2, df_comp["ate"], width,
               label="ATE IPTW", color="steelblue", alpha=0.8)

# IC como barras de erro
ic_lower_err = df_comp["ate"] - df_comp["ic_lower"]
ic_upper_err = df_comp["ic_upper"] - df_comp["ate"]
ax.errorbar(x + width/2, df_comp["ate"],
            yerr=[ic_lower_err, ic_upper_err],
            fmt='none', color='navy', capsize=4, linewidth=1.5)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("Efeito estimado")
ax.set_title("Associação bruta (EDA) vs. ATE IPTW\n(barras de erro = IC 95% bootstrap)",
             fontsize=12)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("../../reports/figures/iptw_v6_comparacao_final.png", dpi=150)
plt.show()